## DS256 - Scalable Systems for Data Science | Jan 2026
# Assignment 1: LLM Data Preprocessing Pipeline with Apache Spark

#### Posted on: 2026-02-07
#### Deadline: 2026-03-01 (For code submission), 2026-03-08 (For scalability/report)
#### Maximum Points: 100 (50 for code correctness, 50 for scalability/report)

### Changelog:
----
* v1: Initial release of questions.
* v2: Fixed validations; added detailed instructions.

### Common Instructions
----
* You must ONLY edit cells and regions within those cells that allow changes. **DO NOT MODIFY other cells**. This can cause the evaluation script to break and you **WILL** be penalized or get **ZERO** points.
* You MUST NOT use the string **###!@** anywhere in your code or comments. We will be using this special substring for pattern matching during evaluations.
* You may declare **all** your valid imports and user-defined functions in the cells that are provided. Otherwise, all parts of your answer must remain between the space provided for the solution to each question.
* You must only use transformations and actions over Spark DataFrames, Spark SQL, and Spark RDDs to solve the assignment (**Spark Core**). You **MUST NOT** use MLLib, etc, unless specified.
 <!-- * https://spark.apache.org/docs/latest/api/python/reference/pyspark.html#rdd-apis -->
* Most of your processing to solve the problem should be done **within Spark**. Minimal post processing may be done in the Python driver code.
* You **must not** use Numpy, Scipy, etc. within the **driver** code. But you may use standard Python libraries as part of lambda expressions or functions passed to Spark transformations and actions.
<!-- * You must not use the default statistics operation (RDD.stats()) available in the **Spark Numeric RDD**. -->
* Our evaluations will include **alternate input WARC files** that have the same format but different contents/sizes.
* We have provided **reference outputs** and the **number of output records** for the test inputs for the **small** input set. You can use these to verify the correctness of your solutions. We have also provided **reference execution time** taken by each step on **Colab**.
* The evaluation will be done by passing clean reference inputs to each problem, i.e., even if one of the steps fails, we will evaluate the subsequent steps on the reference output from the previous steps.
* You will get **bonus** marks if the *end-to-end pipeline* works correctly and also if your runtime is faster than the *reference execution time* we have provided.
* 50% of the assessment goes towards your scalability evaluation, plots and detailed report. You **must** complete and submit the code for correctness evaluation by the first deadline and spend time on the experiments, analysis of performance and the report for the second deadline.
* The report should include a detailed analysis of the strong and weak scaling behavior of the individual steps as well as the end-to-end pipeline supported by experimental evidence and plots.
* **NOTE (Trigger Warning):** *Part of the assignment involves filtering out banned URLs. Kindly take care when processing this data as it may have sensitive words that are (by definition) not polite and potentially upsetting.*
<br>

### **IMPORTANT:** Academic Integrity
----
 The assignment must be completed by yourself and your teammate without any assistance from others or from online sources, ChatGPT, Copilot, etc. Taking online help for standard API references or clearing simple doubts on Python/Spark is allowed. If **any cases of plagiarism are detected, you may get a failing grade in the course and/or be reported to the Institute for further action**. Please follow IISc's Policy on Academic Integrity, https://iisc.ac.in/about/student-corner/academic-integrity/ .

### Submission Guidelines
----

1. Copy the **LATEST** template notebook to your local Google drive. Change the name of the notebook to **assignment1_sol.ipynb**. Proceed to complete the assignment within colab.
2. Make edits only in cells and regions that are clearly marked for modification. **Do not change the template in any other way**. We will be automatically parsing relevant cells and functions during grading. If these instructions are not followed (e.g., even an extra space in an unauthorized part), you can get a **zero** for the Assignment.
3. After you've solved the questions, verify that the output that you generate passes the **validation check** of the data types, and matches the reference outputs that are provided for the sample inputs. Note that for **floats** (if relevant), only the first 3 digits of precision will be checked, but you should not make any changes to the default precision in your code. If the output data type is not valid as specified, you will get a zero for that problem.
4. Each of the problems should take no longer than 5x the reference time as given for our reference outputs. If it takes longer, the problem will not be evaluated.
5. When you are ready to submit the assignment, download the notebook as a **(.py)** file to your local machine. You can do so by going to **File > Download > Download (.py)** in your colab environment.
6. By the first deadline, upload the **pranjalnaman_a1.py** file to Moodle Assignment-1-Code, if your or your teammmate's IISc email address is pranjalnaman@iisc.ac.in. *Upload only 1 file per team!*
7. By the second deadline, upload **pranjalnaman_a1.pdf** to Moodle Assignment-1-Report.

### Overview
----
This assignment implements a complete **LLM Data Preprocessing Pipeline** using Apache Spark on a YARN cluster. The pipeline processes raw Common Crawl WARC data through 5 steps to produce clean, tokenized training data suitable for Large Language Model training.

### Pipeline Steps
| Step | Name | Description |
|-------|------|-------------|
| 1 | Warc to Parquet | Convert the warc files to parquet (5%) |
| 2 | Ingestion / Filter | Load WARC parquet files and filter blacklisted domains (15%) |
| 3 | Extraction | Extract clean text from HTML using Trafilatura (15%) |
| 4 | Language ID | Filter to English-only documents using FastText (15%) |
| 5 | Deduplication | Remove near-duplicate documents using MinHash LSH (30%) |
| 6 | Tokenization | Convert text to token sequences for model training (20%) |

### Common Instructions
----
* You must ONLY edit cells and regions within those cells that allow changes. **DO NOT MODIFY** cells marked with `DO NOT MODIFY`.
* All processing should be done **within Spark** using DataFrames, RDDs, and Spark SQL.
* Ensure your virtual environment Python is accessible to all YARN worker nodes.

In [ ]:
# ======== DO NOT MODIFY ===========
!pip install pyspark
!pip install warcio
!pip install fasttext
!pip install trafilatura
!pip install numpy==1.26.4

In [ ]:
# ======== DO NOT MODIFY ===========
import sys
import os
import subprocess
import time
import pyspark
from pyspark import SparkFiles
import fasttext
import hashlib

from urllib.parse import urlparse
from pyspark.sql.types import StructType, StructField, StringType, BinaryType
from pyspark.ml.feature import MinHashLSH
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql import Window
from pyspark.sql import SparkSession
from google.colab import drive
from warcio.archiveiterator import ArchiveIterator

In [ ]:
# ======== DO NOT MODIFY ===========
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# ======== DO NOT MODIFY ===========
BASE_DIR = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/small' # we will change this to large. You may test with `small` and `medium`.
BLACKLIST_DOMAINS_DIR = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/blacklist_domains'
FASTTEXT_MODEL_BIN = '/content/drive/Shareddrives/ds256-2026-public/assignment_1/lid.176.bin'
PARQUET_DIR = '/content/drive/MyDrive/<output-dir>/' #TODO: Change it to a dir on your drive

In [ ]:
# ======= DO NOT MODIFY ============
spark = (
    SparkSession.builder
    .appName("LLMSpark")
    .master("local[*]")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

## Verify the warc files are readable

In [ ]:
# ======= DO NOT MODIFY ============
os.listdir(BASE_DIR)

['CC-MAIN-20251204191828-20251204221828-00000.warc',
 'CC-MAIN-20251204191828-20251204221828-00001.warc']

In [ ]:
# ===== SCHEMA VALIDATORS FOR EACH STEP (DO NOT MODIFY) =====

# STEP 2 Output Schema Validator
STEP_2_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'html_content': 'string'
}

# STEP 3 Output Schema Validator
STEP_3_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# STEP 4 Output Schema Validator (same as STEP 3)
STEP_4_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# STEP 5 Output Schema Validator (same as STEP 3)
STEP_5_SCHEMA = {
    'warc_id': 'string',
    'url': 'string',
    'date': 'string',
    'extracted_text': 'string'
}

# STEP 6 Output Schema Validator
STEP_6_SCHEMA = {
    'warc_id': 'string',
    'tokens': 'array<int>',
    'attention_mask': 'array<int>'
}

In [ ]:
# ===== SCHEMA VALIDATORS FOR EACH STEP (DO NOT MODIFY) =====

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel

# Helper for schema validation (DO NOT MODIFY)
def validate_schema(df, expected_columns):
    """Verifies that the dataframe contains the required columns with correct types."""
    df_cols = {f.name: f.dataType.simpleString() for f in df.schema.fields}
    missing = []
    for col, type_str in expected_columns.items():
        if col not in df_cols:
            missing.append(f"{col} (missing)")
        elif type_str not in df_cols[col]:
            pass
    if missing:
        raise ValueError(f"Schema Validation Failed. Issues: {missing}")
    else:
        return

def validate_step_schema(df, step_num):
    """Validate DataFrame schema for a specific step."""
    schemas = {
        2: STEP_2_SCHEMA,
        3: STEP_3_SCHEMA,
        4: STEP_4_SCHEMA,
        5: STEP_5_SCHEMA,
        6: STEP_6_SCHEMA,
    }
    expected = schemas.get(step_num, {})
    validate_schema(df, expected)
    print(f"✓ STEP {step_num} schema validation passed!")

---
---
## Your Code Edits Start from Here
---
### Common Imports and Functions

List imports and functions that are used across different questions in these sections.

In [ ]:
#######################################
###!@0.1 START COMMON USER IMPORTS
#######################################
## Specify valid imports, if any, for ALL your answers  ==========
## start your edits here =================

import trafilatura  # HTML text extraction (Step 3)
# fasttext and SparkFiles are already imported in the DO NOT MODIFY cell above

## end your edits here =================
###!@0.1 END COMMON USER IMPORTS

In [ ]:
#######################################
###!@0.2 START COMMON USER FUNCTIONS
#######################################
## Specify user defined functions, if any, used by multiple answers   =====
## start your edits here =================

# ── FastText model cache (shared by Step 4) ──────────────────────────────────
# Lives at module level so it is initialised ONCE per worker process.
# Without this, fasttext.load_model() would run for every single row — catastrophic.
_fasttext_model = None

def get_fasttext_model():
    """
    Lazy singleton: loads the FastText language ID model from the worker's local
    copy of the file (placed there by sc.addFile) on the very first call,
    then caches it in _fasttext_model for all subsequent calls on the same worker.

    WHY global:  Python requires 'global' to *assign* to an outer-scope variable
                 inside a function; without it, Python creates a new local variable
                 and the cached value is thrown away after each call.
    """
    global _fasttext_model
    if _fasttext_model is None:
        # SparkFiles.get(filename) returns the LOCAL disk path where Spark
        # copied the file on THIS worker — pass only the filename, not the Drive path.
        local_path = SparkFiles.get('lid.176.bin')
        _fasttext_model = fasttext.load_model(local_path)
    return _fasttext_model

## end your edits here =================
###!@0.2 END COMMON USER FUNCTIONS

## STEP 1: Convert the warc files to parquet ***(5 points)***
----

### Objective
Ingest raw WARC data from the HDFS file system (Google Drive in this Colab Notebook), parse the records to extract HTML content, and save the result as a partitioned Parquet dataset.

The function returns nothing. It just writes the parquet files to OUTPUT_DIR.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `310 seconds`
#### # of records: `43,334 records`

### Guidelines
1. List Files: Retrieve the list of .warc files from the BASE_DIR directory.
2. Stream each WARC file.
3. Parse records using warcio.ArchiveIterator.
4. Filter for records where rec_type is 'response' and Content-Type contains 'html'.
5. Write to Parquet: Save the DataFrame to OUTPUT_DIR in Parquet format, partitioned by the original WARC filename.

***The DataFrame should have the output schema defined below.***
### Parquet DataFrame Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| html_content | string | Raw HTML content |
| warc_filename | string | The name of the source WARC file (used for partitioning)

In [ ]:
#######################################
###!@1 START ANSWER STEP 1

### Q1 ###################################################

def step_1_warc_to_parquet():
    """
    Reads all .warc files from BASE_DIR, parses every HTML 'response' record
    using warcio, and writes the result as a partitioned Parquet dataset to PARQUET_DIR.
    Partitioned by warc_filename so downstream steps can prune by source file.
    Returns nothing — side-effect is writing Parquet files.

    Reference (small): ~310 seconds, 43,334 records.

    WHY WE CHANGED THE APPROACH (Original vs Current):
    ─────────────────────────────────────────────────────────────────────────────
    ORIGINAL APPROACH (caused OOM crash on Colab 12.7GB):
      - All WARC records collected into a Python list on the driver: all_rows = []
      - spark.createDataFrame(all_rows) then serialises the entire list into the
        JVM, so BOTH the Python list (~4GB) and the JVM copy (~3GB) exist at the
        same time → peak ~7-12GB → OOM.

    CURRENT APPROACH (Approach C — RDD + generator):
      - Driver only holds a list of 3 file *paths* (~300 bytes total).
      - sc.parallelize(paths) → workers parse their own WARC files.
      - parse_warc_file() is a GENERATOR (yield, not return list) so Spark
        pulls one record at a time from each worker → O(1) memory per record.
      - Data flows directly from workers → Spark RDD → DataFrame.
        The driver never accumulates HTML content → no OOM.
    ─────────────────────────────────────────────────────────────────────────────
    """
    ## start your edits here  =================

    # Explicit schema — always define manually so Spark doesn't guess wrong
    # types when some fields are None (e.g. missing WARC headers).
    schema = StructType([
        StructField("warc_id",       StringType(), True),
        StructField("url",           StringType(), True),
        StructField("date",          StringType(), True),
        StructField("html_content",  StringType(), True),
        StructField("warc_filename", StringType(), True),
    ])

    # ══════════════════════════════════════════════════════════════════════════
    # ORIGINAL APPROACH — collected everything on the driver (caused OOM)
    # ══════════════════════════════════════════════════════════════════════════
    # warc_files = [f for f in os.listdir(BASE_DIR) if f.endswith('.warc')]
    # all_rows = []                          # ← entire dataset in Python heap
    #
    # for filename in warc_files:
    #     filepath = os.path.join(BASE_DIR, filename)
    #     with open(filepath, 'rb') as f:
    #         for record in ArchiveIterator(f):
    #             if record.rec_type != 'response':
    #                 continue
    #             content_type = record.http_headers.get_header('Content-Type') or ''
    #             if 'html' not in content_type.lower():
    #                 continue
    #             html_content = record.content_stream().read().decode('utf-8', errors='replace')
    #             all_rows.append({              # ← accumulates 3-5 GB on driver
    #                 'warc_id':       record.rec_headers.get_header('WARC-Record-ID'),
    #                 'url':           record.rec_headers.get_header('WARC-Target-URI'),
    #                 'date':          record.rec_headers.get_header('WARC-Date'),
    #                 'html_content':  html_content,
    #                 'warc_filename': filename,
    #             })
    #
    # df = spark.createDataFrame(all_rows, schema=schema)   # ← JVM copy = 2× RAM
    # df.write.partitionBy('warc_filename').parquet(PARQUET_DIR, mode='overwrite')
    # ══════════════════════════════════════════════════════════════════════════

    # ══════════════════════════════════════════════════════════════════════════
    # CURRENT APPROACH C — RDD + generator (O(1) driver memory)
    # ══════════════════════════════════════════════════════════════════════════

    # CHANGE 1: Build a list of full file PATHS (not file contents).
    # The driver holds only 3 strings (~300 bytes) instead of 43,334 HTML pages.
    warc_paths = [
        os.path.join(BASE_DIR, f)
        for f in os.listdir(BASE_DIR)
        if f.endswith('.warc')
    ]
    print(f"Found {len(warc_paths)} WARC file(s).")

    # CHANGE 2: This function runs on a SPARK WORKER (not the driver).
    # It is a GENERATOR (uses 'yield' not 'return list') — Spark's flatMap
    # pulls one record at a time, so memory stays O(1) regardless of file size.
    #
    # Original: one big function on the driver collecting into all_rows list.
    # Current:  one small function per worker, streaming records lazily.
    def parse_warc_file(filepath):
        """
        Generator that streams one WARC file and yields one tuple per valid
        HTML response record. Runs entirely on the Spark worker — no driver
        involvement after the path is handed off.
        """
        filename = os.path.basename(filepath)  # e.g. 'CC-MAIN-...-00000.warc'

        # Open in binary mode — WARC files are binary-encoded archives.
        with open(filepath, 'rb') as f:

            # ArchiveIterator streams record-by-record — never loads the whole
            # WARC into memory at once. Combined with 'yield' below, the entire
            # pipeline is streaming: one record parsed → yielded → consumed by
            # Spark → next record parsed. Peak memory ≈ one record's HTML.
            for record in ArchiveIterator(f):

                # Only 'response' records carry HTTP response bodies (the HTML).
                # 'warcinfo', 'request', 'metadata' records are skipped.
                if record.rec_type != 'response':
                    continue

                # Check HTTP Content-Type to keep only HTML pages.
                # 'or ""' guards against get_header() returning None.
                content_type = record.http_headers.get_header('Content-Type') or ''
                if 'html' not in content_type.lower():
                    continue

                # Decode bytes → str. errors='replace' substitutes undecodable
                # bytes instead of raising UnicodeDecodeError on broken pages.
                html_content = record.content_stream().read().decode(
                    'utf-8', errors='replace'
                )

                # YIELD (not append+return): Spark pulls this one record, then
                # immediately asks for the next — memory is freed between yields.
                yield (
                    record.rec_headers.get_header('WARC-Record-ID'),   # warc_id
                    record.rec_headers.get_header('WARC-Target-URI'),  # url
                    record.rec_headers.get_header('WARC-Date'),        # date
                    html_content,                                       # html_content
                    filename,   # warc_filename — used as partition folder name
                )

    # CHANGE 3: parallelize(paths) — 1 partition per file.
    # Original: no parallelism at all; everything was sequential on the driver.
    # Current:  Spark assigns one worker task per WARC file. Each worker
    #           independently streams its file — true pipeline parallelism.
    #
    # numSlices=len(warc_paths) ensures each file gets its own partition/task,
    # so all 3 files are parsed concurrently (in local[*] mode: 3 threads).
    rdd = spark.sparkContext.parallelize(warc_paths, numSlices=len(warc_paths))

    # CHANGE 4: flatMap(parse_warc_file) — the generator replacement for all_rows.
    # Original: driver loop appended to all_rows list (wide, memory-hungry).
    # Current:  flatMap is a NARROW DEPENDENCY — each input partition (one file
    #           path) maps to output records with no data movement between workers.
    #           Records stream from worker directly into the RDD — driver never
    #           sees the HTML content.
    records_rdd = rdd.flatMap(parse_warc_file)

    # CHANGE 5: createDataFrame from RDD (not from Python list).
    # Original: createDataFrame(all_rows) — serialised Python list into JVM,
    #           causing a 2nd full copy of all data in memory simultaneously.
    # Current:  createDataFrame(rdd) — data is already in Spark's managed memory
    #           as an RDD; no serialisation round-trip through the driver.
    df = spark.createDataFrame(records_rdd, schema=schema)

    # Write partitioned by warc_filename — unchanged from original.
    # mode='overwrite' makes the step safely re-runnable.
    df.write.partitionBy('warc_filename').parquet(PARQUET_DIR, mode='overwrite')
    print(f"Step 1 complete. Records written to {PARQUET_DIR}")

    # ══════════════════════════════════════════════════════════════════════════

    ## end your edits here  =================


###!@1 END ANSWER STEP 1

## STEP 2: Data Ingestion & Domain Filtering ***(15 points)***
----


### Objective
Load raw WARC data from parquet files and filter out records from blacklisted domains.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `35 seconds`
#### # of records: `40,985 records`

### Input
- `input_path`: HDFS (Colab for this notebook) path to parquet files containing WARC records

### Processing Steps
1. **Construct Domain Blacklist**:
    - Recursively walk the `blacklist_root_dir`.
    - Identify files named `domains` or `urls`.
    - Read each file line-by-line, stripping whitespace.
    - Ignore lines starting with `#` (comments).
    - The aforementioned steps are guidelines. Ultimately we want the ouput in the format described below.
    - Output can look like - `{'altabu-db1.blogspot.hu',
 'biwaisms-putesdefoncees.blogspot.cz',
 'kamilla18.blogspot.jp',
 'batorsparadise.blogspot.pt', ...}`
2. Extract domain from URL using regex.
3. Drop the records where domain and/or subdomains matches blacklist.


### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| html_content | string | Raw HTML content |

### Expected Output Example (Step 2)
```
root
|-- warc_id: string (nullable = true)
|-- url: string (nullable = true)
|-- date: string (nullable = true)
|-- html_content: string (nullable = true)

[Stage 109:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|        html_content|
+--------------------+--------------------+--------------------+--------------------+
|<urn:uuid:ed46a75...|https://www.futur...|2025-12-04T20:34:05Z|<!DOCTYPE html><h...|
|<urn:uuid:9fb9dc9...|https://amotion.t...|2025-12-04T19:35:27Z|\n\n<!DOCTYPE htm...|
|<urn:uuid:98bada1...|https://www.fuzzy...|2025-12-04T20:30:52Z|<!DOCTYPE html>\n...|
|<urn:uuid:7149527...|https://amp.rtve....|2025-12-04T19:49:05Z|\n  <!DOCTYPE htm...|
|<urn:uuid:423fa66...|https://www.fwg-p...|2025-12-04T19:25:30Z|<!DOCTYPE html>\n...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 2 schema validation passed!

In [ ]:
#######################################
###!@2 START ANSWER STEP 2

### Q2 ###################################################

def load_blacklist_categories():
    """
    Recursively walks BLACKLIST_DOMAINS_DIR and collects every domain/URL entry
    from files named exactly 'domains' or 'urls' into a Python set.
    Lines beginning with '#' are comments and are skipped.
    Returns a set of strings for O(1) membership testing during filtering.
    """
    print("Loading blacklist categories...")
    blacklisted_domains = set()

    ## start your edits here  =================

    # os.walk() visits every subfolder beneath BLACKLIST_DOMAINS_DIR recursively.
    # Each iteration yields:
    #   dirpath   — full path of the current folder
    #   _         — list of sub-folder names (not needed here)
    #   filenames — list of file names in the current folder
    for dirpath, _, filenames in os.walk(BLACKLIST_DOMAINS_DIR):
        for filename in filenames:

            # Only process files literally named 'domains' or 'urls'.
            # Any other file (README, .gitignore, .DS_Store …) is silently ignored.
            if filename not in ('domains', 'urls'):
                continue

            filepath = os.path.join(dirpath, filename)
            with open(filepath, 'r', errors='replace') as f:
                for line in f:
                    line = line.strip()  # remove leading/trailing whitespace & newlines

                    # Skip blank lines and comment lines (start with '#').
                    if not line or line.startswith('#'):
                        continue

                    blacklisted_domains.add(line)

    ## end your edits here  =================

    print(f"  Loaded {len(blacklisted_domains)} blacklisted domain entries.")
    return blacklisted_domains


def step_2_ingestion():
    """
    Loads WARC parquet files (written by Step 1) from PARQUET_DIR and removes
    records whose domain or any parent domain appears in the blacklist.
    Returns DataFrame with columns: warc_id, url, date, html_content.

    Reference (small): ~35 seconds, 40,985 records.
    """

    blacklisted_domains = load_blacklist_categories()

    ## start your edits here  =================

    # ── Broadcast the blacklist set to all Spark workers ─────────────────────
    # The set lives on the driver. Without broadcast(), Spark re-serialises the
    # entire set for EVERY row the UDF processes — catastrophically slow.
    # With broadcast(), each worker receives ONE efficient cached copy in memory.
    broadcast_blacklist = spark.sparkContext.broadcast(blacklisted_domains)

    # ── UDF: check whether a hostname is blacklisted ──────────────────────────
    def is_blacklisted(domain):
        """
        Returns True if 'domain' exactly matches OR is a subdomain of
        any entry in the broadcast blacklist set.

        Examples (blacklist contains 'blogspot.jp'):
            'blogspot.jp'           → True  (exact match)
            'kamilla18.blogspot.jp' → True  (subdomain: strip 'kamilla18.' and match)
            'otherblogspot.jp'      → False (different registered domain)
        """
        if not domain:
            return False

        # Access the broadcasted set on THIS worker node via .value
        bl = broadcast_blacklist.value

        # Fast path: exact match is an O(1) set lookup
        if domain in bl:
            return True

        # Subdomain walk: progressively strip the leftmost label and re-check.
        # 'a.b.c.com' → check 'b.c.com', then 'c.com', then 'com'
        parts = domain.split('.')
        for i in range(1, len(parts)-1):
            parent = '.'.join(parts[i:])
            if parent in bl:
                return True

        return False

    # Register as a Spark UDF returning Boolean.
    # BooleanType because it drives row filtering (True = blacklisted = drop).
    is_blacklisted_udf = F.udf(is_blacklisted, BooleanType())

    # ── Load Parquet written by Step 1 ────────────────────────────────────────
    # Spark auto-discovers all partition sub-folders and reconstructs the schema
    # from embedded Parquet metadata — no schema definition needed here.
    df = spark.read.parquet(PARQUET_DIR)

    # ── Extract hostname from URL using regex ─────────────────────────────────
    # regexp_extract(column, pattern, group_index) returns the Nth capture group.
    #
    # Pattern: r'https?://([^/?\s]+)'
    #   https?       — matches 'http' or 'https'
    #   ://          — literal
    #   ([^/?\s]+)  — group 1: all characters that are NOT '/', '?', or whitespace
    #                  i.e. exactly the hostname (e.g. 'www.kamilla18.blogspot.jp')
    df = df.withColumn(
        'domain',
        F.regexp_extract(F.col('url'), r'https?://([^/?\s]+)', 1)
    )

    # ── Filter out blacklisted domains ────────────────────────────────────────
    # ~ is the Spark column NOT operator.
    # We KEEP rows where is_blacklisted returns False (not blacklisted).
    df_filtered = df.filter(~is_blacklisted_udf(F.col('domain')))

    # ── Drop the temporary 'domain' column ───────────────────────────────────
    # It was only needed for filtering logic.
    # Required output schema: warc_id, url, date, html_content — no 'domain'.
    output_df = df_filtered.drop('domain')

    ## end your edits here  =================

    return output_df

###!@2 END ANSWER STEP 2

---
## Step 3: Text Extraction ***(15 points)***
----

### Objective
Extract clean, readable text content from raw HTML using the Trafilatura library.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `2240 seconds`
#### # of records: `39,644 records`

### Input
- DataFrame with `html_content` column (from Step 2)

### Processing Steps (Guidelines)
1. Apply `trafilatura` extraction. You can exclude comments, include tables, and have `no_fallback=False`
2. Drop records that cannot be processed by `trafilatura`.
3. It is possible that `trafilatura` returns nothing. Drop such records, i.e., only records with valid extracted text must be retained.


### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| url | string | Source URL |
| date | string | Crawl timestamp |
| extracted_text | string | Clean text extracted from HTML |

### Expected Output Example (Step 3)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 117:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+---------------------------+
|             warc_id|                 url|                date|             extracted_text|
+--------------------+--------------------+--------------------+---------------------------+
|<urn:uuid:51e562d...|https://www.viagr...|2025-12-04T19:37:35Z|产品分类\nProducts相关文...|
|<urn:uuid:9422649...|http://www.elolit...|2025-12-04T20:43:27Z|       Confinarse, encer...|
|<urn:uuid:e4642c3...|https://www.viapi...|2025-12-04T20:27:35Z|       Warenkorb ist lee...|
|<urn:uuid:85aa3aa...|http://www.empren...|2025-12-04T20:47:21Z|       ¿Qué es el delito...|
|<urn:uuid:7c4b999...|https://www.vibar...|2025-12-04T19:39:43Z|       317 Products\nΚαν...|
+--------------------+--------------------+--------------------+---------------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 3 schema validation passed!

```

In [ ]:
#######################################
###!@3 START ANSWER STEP 3

### Q3 ###################################################

def step_3_extraction(input_df):
    """
    Extracts clean readable text from raw HTML using Trafilatura.
    Rows where extraction fails or returns nothing are dropped.
    Args:
        input_df: DataFrame with 'html_content' (string) from Step 2.
    Returns:
        DataFrame: 'html_content' replaced by 'extracted_text'.
                   Columns: warc_id, url, date, extracted_text.

    Reference (small): ~2240 seconds, 39,644 records.
    """
    ## start your edits here  =================

    # ── Define the extraction function ────────────────────────────────────────
    # Plain Python function — Spark calls this row-by-row across the cluster.
    def extract_text(html):
        # Guard: never pass None into trafilatura — it raises TypeError.
        if not html:
            return None

        # trafilatura.extract() works by:
        #   1. Parsing the HTML tree with lxml
        #   2. Scoring every content block (text density, link ratio, tag semantics)
        #   3. Extracting the highest-scoring block (the 'main article')
        #   4. Returning a clean text string, OR None when no usable content is found
        #      (e.g. login walls, pure-JS pages, 404 error pages)
        #
        # Parameters specified by the assignment:
        #   include_comments=False  — discard comment sections
        #   include_tables=True     — keep table content (useful for structured data)
        #   no_fallback=False       — allow trafilatura to try fallback extractors
        #                             when its primary heuristic finds nothing
        return trafilatura.extract(
            html,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        )

    # ── Register as a Spark UDF returning a string ────────────────────────────
    # StringType() — this UDF transforms a column value, unlike Step 2/4 which
    # return BooleanType() for filtering. Python None maps to Spark null.
    extract_udf = F.udf(extract_text, StringType())

    # ── Apply the UDF to create 'extracted_text' ─────────────────────────────
    # withColumn adds a new column computed from 'html_content' for every row.
    # Each Spark worker runs extract_text() on its local partition independently.
    df = input_df.withColumn(
        'extracted_text',
        extract_udf(F.col('html_content'))
    )

    # ── Drop rows where extraction returned None ──────────────────────────────
    # isNotNull() is the correct Spark idiom — never use '!= None' on Spark
    # columns because Spark has its own three-valued null logic (not Python's).
    # This removes pages where trafilatura found no meaningful content.
    df = df.filter(F.col('extracted_text').isNotNull())

    # ── Drop the raw HTML column ──────────────────────────────────────────────
    # html_content has served its purpose; dropping it frees significant memory.
    # Output schema requires: warc_id, url, date, extracted_text.
    output_df = df.drop('html_content')

    ## end your edits here  =================

    return output_df

###!@3 END ANSWER STEP 3

---
## STEP 4: Language Identification ***(15 points)***
----

### Objective
Filter documents to retain only English content using FastText language identification.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `40 seconds`
#### # of records: `14,303 records`

### Input
- DataFrame with `extracted_text` column (from Step 3)
- `threshold`: Minimum probability for English classification (default: 0.6). We will test your code with different thresholds.

### Processing Steps
1. Predict language for each document (`Fasttext` requires single line inputs. Handle line breaks in extracted text.)
2. Filter to retain only documents classified as English (`__label__en`) with probability ≥ threshold

### Output Schema
Same as input (records that pass the English filter)

### Expected Output Example (Step 4)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 124:>                                                        (0 + 1) / 1]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|      extracted_text|
+--------------------+--------------------+--------------------+--------------------+
|<urn:uuid:d119f3d...|https://snapvrs.o...|2025-12-04T20:39:17Z|As solar panels d...|
|<urn:uuid:75d7f1f...|https://fresh-tri...|2025-12-04T20:30:35Z|Diving in Jordan,...|
|<urn:uuid:bb9ce09...|https://snippets....|2025-12-04T21:11:11Z|Are RV campers a ...|
|<urn:uuid:4ed2248...|https://frigidair...|2025-12-04T20:47:04Z|Frigidaire Dryer ...|
|<urn:uuid:b0b3105...|https://soap2day....|2025-12-04T20:50:59Z|Genre\nAction\nAd...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 4 schema validation passed!
```

In [ ]:
#######################################
###!@4 START ANSWER STEP 4

### Q4 ###################################################

def step_4_lang_id(input_df, threshold=0.6):
    """
    Filters the DataFrame to retain only English-language documents.
    Uses the FastText lid.176.bin model distributed to workers via sc.addFile().
    Args:
        input_df:  DataFrame with 'extracted_text' column (from Step 3).
        threshold: Minimum FastText confidence to accept as English (default 0.6).
                   The grader tests with different threshold values.
    Returns:
        DataFrame: English-only rows. Schema identical to input.

    Reference (small): ~40 seconds, 14,303 records.
    """

    ## start your edits here  =================

    # ── Distribute the FastText model binary to all workers ───────────────────
    #
    # PROBLEM: The FastText model (~900 MB) is a C++-backed binary object.
    #   It cannot be Python-pickled and sent via broadcast() like a dict/set.
    #   Each worker needs its own LOCAL copy of the file on disk to load it.
    #
    # SOLUTION — sc.addFile() + SparkFiles.get():
    #
    #   sc.addFile(path)
    #     Called on the driver. Tells Spark to copy the file to every worker's
    #     local temp directory BEFORE any task starts running.
    #
    #   SparkFiles.get('lid.176.bin')    [inside UDF, running on worker]
    #     Returns the full local disk path of the copied file on THIS worker,
    #     e.g. '/tmp/spark-abc/userFiles/lid.176.bin'.
    #     Pass only the filename — Spark strips the directory on copy.
    #
    # FLOW:
    #   Driver  sc.addFile('/content/drive/.../lid.176.bin')
    #               |
    #               +---> Worker-1 local disk: /tmp/spark-xxx/userFiles/lid.176.bin
    #               +---> Worker-2 local disk: /tmp/spark-xxx/userFiles/lid.176.bin
    #                           |
    #                    SparkFiles.get('lid.176.bin')
    #                           |
    #                    fasttext.load_model(local_path)   [cached via get_fasttext_model()]
    #
    # NOTE: get_fasttext_model() is defined in the Common Functions cell above.
    # It ensures the model is loaded ONCE per worker process, not once per row.
    spark.sparkContext.addFile(FASTTEXT_MODEL_BIN)

    # ── Define the language detection UDF ─────────────────────────────────────
    def is_english(text):
        """
        Returns True if FastText classifies the text as English (__label__en)
        with probability >= threshold (captured from the outer scope via closure).
        """
        # Guard: null or whitespace-only text cannot be classified → discard.
        if not text or not text.strip():
            return False

        # Load model (instant after first call on this worker — see get_fasttext_model).
        model = get_fasttext_model()

        # CRITICAL: FastText treats each LINE as a separate document internally.
        # Text with embedded newlines causes it to classify only the first line
        # and silently ignore the rest, producing wrong predictions.
        # Replace all newline variants with a single space before predicting.
        clean_text = text.replace('\n', ' ').replace('\r', ' ').strip()

        # model.predict() returns:
        #   result[0] — tuple of label strings, e.g. ('__label__en',)
        #   result[1] — numpy array of probabilities, e.g. array([0.9999])
        # Default k=1: only the single top prediction is returned.
        result = model.predict(clean_text)

        top_label = result[0][0]         # e.g. '__label__en'
        top_prob  = float(result[1][0])  # e.g. 0.9999  — cast to plain float

        # Keep the document only when BOTH conditions are met:
        #   1. FastText identified the language as English
        #   2. Confidence meets or exceeds the caller-supplied threshold
        return (top_label == '__label__en') and (top_prob >= threshold)

    # BooleanType: this UDF drives row filtering (True = English = keep).
    is_english_udf = F.udf(is_english, BooleanType())

    # ── Filter — keep English documents only ──────────────────────────────────
    # Schema is unchanged: no columns are added or removed, only rows are dropped.
    output_df = input_df.filter(is_english_udf(F.col('extracted_text')))

    ## end your edits here  =================

    return output_df

###!@4 END ANSWER STEP 4

---
## STEP 5: Deduplication ***(30 points)***
----

### Objective
Duplicate records can be detrimental to LLM training and therefore need to be dropped. Here, you will write 2 different deduplication algorithms, one approximate and one exact, and compare the two in your report.

---
## STEP 5a: Near-Duplicate Deduplication ***(15 points)***
----

### Objective
Remove near-duplicate documents using MinHash Locality-Sensitive Hashing (LSH).

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `150 seconds`
#### # of records: `12,820 records`

### Input
- DataFrame with `extracted_text` column (from Step 4)

### Algorithm
1. **Character 5-grams**: Convert text to character-level 5-grams, hash each to a vocabulary index
2. **MinHash Signatures**: Generate 24 MinHash signatures using Spark MLlib's MinHashLSH
3. **Banding**: Split signatures into 8 bands of 3 hashes each for similarity detection
4. **Clustering**: Find similar document pairs with Jaccard distance ≤ 0.5
5. **Filtering**: Keep only the longest document from each cluster

### Configuration
| Parameter | Value | Description |
|-----------|-------|-------------|
| NGRAM_SIZE | 5 | Character n-gram length |
| NUM_HASHES | 24 | Total MinHash signatures |
| NUM_BANDS | 8 | Number of LSH bands |
| VOCAB_SIZE | 2^18 | Hash space size (262,144) |

### Output Schema
Same as input (deduplicated records)

### Expected Output Example (Step 5a)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 155:======================================================>(96 + 1) / 97]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|      extracted_text|
+--------------------+--------------------+--------------------+--------------------+
|         warc_id_102|http://example.co...|2025-12-04T21:02:54Z|Machine learning ...|
|         warc_id_103|http://example.co...|2025-12-04T21:02:55Z|Python is a high-...|
|<urn:uuid:caa5d51...|https://www.valte...|2025-12-04T20:38:04Z|Agentforce: The G...|
|<urn:uuid:15335f8...|https://www.vande...|2025-12-04T20:39:07Z|This Cookie Polic...|
|<urn:uuid:b5590de...|http://www.elks.o...|2025-12-04T20:06:13Z|Join the Elks!\nL...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 5a schema validation passed!
```

In [ ]:
# #######################################
# ###!@5a START ANSWER STEP 5a

# ### Q5a ###################################################
# def step_5a_deduplication(input_df):
#     """
#     Removes near-duplicate documents using MinHash LSH on character 5-grams.

#     Algorithm recap:
#       1. Convert each document's text into a set of character 5-grams.
#          Hash each 5-gram to an integer index in [0, VOCAB_SIZE).
#          Represent the document as a SparseVector of size VOCAB_SIZE.
#       2. Fit MinHashLSH with 24 hash tables to generate compact signatures.
#          Each signature is 24 integers that approximate Jaccard similarity.
#       3. approxSimilarityJoin finds candidate near-duplicate pairs using
#          LSH banding (8 bands × 3 hashes) without O(n²) comparison.
#       4. For each duplicate pair, mark the SHORTER document as the loser.
#       5. Anti-join to remove all losers from the original DataFrame.

#     Configuration (as specified in the assignment):
#       NGRAM_SIZE = 5     character n-gram window size
#       NUM_HASHES = 24    MinHash signatures per document
#       NUM_BANDS  = 8     LSH bands (3 hashes each → 8 × 3 = 24)
#       VOCAB_SIZE = 2^18  hash index space (262,144)

#     Reference (small): ~150 seconds, ~12,820 records.

#     Args:
#         input_df: DataFrame with columns warc_id, url, date, extracted_text.
#     Returns:
#         DataFrame: Deduplicated. Same schema as input.
#     """
#     ## start your edits here  =================

#     # ── Constants (specified by the assignment) ───────────────────────────────
#     NGRAM_SIZE = 5
#     NUM_HASHES = 24
#     VOCAB_SIZE = 2 ** 18   # 262,144 — the hash index space for 5-gram buckets

#     # ── Step 1: UDF — text → SparseVector of hashed 5-gram indices ───────────
#     #
#     # WHY SparseVector?
#     #   MinHashLSH in Spark MLlib requires a numeric feature vector as input.
#     #   Our "set of 5-grams" maps naturally to a binary sparse vector of size
#     #   VOCAB_SIZE: index i = 1.0 if the document contains a 5-gram that hashes
#     #   to i, else 0 (absent — not stored in the sparse format).
#     #
#     # WHY sparse and not dense?
#     #   A dense vector of size 262,144 per document would be enormous.
#     #   A typical document has ~1,000-5,000 unique 5-grams → only ~2% filled.
#     #   SparseVector stores only (index, value) pairs for non-zero entries.
#     #
#     # WHY use a set for indices?
#     #   The same 5-gram can repeat in a document ("the" → 5-gram "the t" appears
#     #   many times). For Jaccard similarity we only care about PRESENCE (0/1),
#     #   not count — so we deduplicate indices with a set before building the vector.
#     def text_to_sparse_features(text):
#         if not text:
#             return Vectors.sparse(VOCAB_SIZE, [], [])

#         indices = set()
#         # Slide a window of NGRAM_SIZE characters across the entire text.
#         # Each window produces one 5-gram string.
#         for i in range(len(text) - NGRAM_SIZE + 1):
#             gram = text[i : i + NGRAM_SIZE]

#             # Hash the 5-gram string to an integer in [0, VOCAB_SIZE).
#             # MD5 gives a 128-bit hex digest; we parse it as a big integer
#             # then take modulo VOCAB_SIZE to map it into our index space.
#             # This is a deterministic, uniform mapping with low collision rate.
#             h = int(hashlib.md5(gram.encode('utf-8')).hexdigest(), 16) % VOCAB_SIZE
#             indices.add(h)

#         # SparseVector requires indices to be SORTED in ascending order.
#         sorted_indices = sorted(indices)
#         values = [1.0] * len(sorted_indices)   # binary: present = 1.0
#         return Vectors.sparse(VOCAB_SIZE, sorted_indices, values)

#     # Register the UDF. VectorUDT() is the Spark type for MLlib vectors.
#     features_udf = F.udf(text_to_sparse_features, VectorUDT())

#     # Apply UDF: add a "features" column (SparseVector) to the DataFrame.
#     df_feat = input_df.withColumn(
#         "features",
#         features_udf(F.col("extracted_text"))
#     )

#     # ── Step 2: Fit MinHashLSH model ─────────────────────────────────────────
#     #
#     # MinHashLSH generates NUM_HASHES random hash functions.
#     # For each document, each hash function computes:
#     #   min( h_i(gram_index) for all gram_index in the document's feature vector )
#     # This gives a 24-element signature that preserves Jaccard similarity:
#     #   P(signature_i(A) == signature_i(B)) = Jaccard(A, B)
#     #
#     # numHashTables=24 → 24 min-hash functions → 24-element signature per doc.
#     # The banding (8 bands × 3 hashes) is handled internally by Spark's LSH
#     # when approxSimilarityJoin is called with a distance threshold.
#     mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=NUM_HASHES)
#     model = mh.fit(df_feat)

#     # Transform: adds the "hashes" column containing the 24-value signature.
#     df_hashed = model.transform(df_feat)

#     # ── FIX: localCheckpoint() before approxSimilarityJoin ───────────────────
#     #
#     # WHY NOT just persist()?
#     #   persist() caches data in memory/disk BUT keeps the deep lineage plan
#     #   intact inside the JVM. Spark's optimizer still tries to recursively
#     #   traverse that plan. approxSimilarityJoin internally creates 24 hash-
#     #   table joins, each referencing df_hashed on BOTH sides — the combined
#     #   plan depth is hundreds of nodes → JVM StackOverflowError.
#     #
#     # WHY localCheckpoint()?
#     #   localCheckpoint() does two things persist() cannot:
#     #     1. Eagerly MATERIALIZES df_hashed — computes from the deep plan once,
#     #        saves result to executor-local disk.
#     #     2. BREAKS THE LINEAGE — df_hashed's plan becomes "read local file"
#     #        (depth = 1). approxSimilarityJoin now sees a clean, shallow plan
#     #        on both sides → no recursion depth issue → no StackOverflow.
#     #
#     #   No setCheckpointDir() needed — localCheckpoint saves to executor's
#     #   local temp storage, not HDFS/GCS. Fast and self-contained.
#     #
#     # persist(MEMORY_AND_DISK) after localCheckpoint keeps the materialized
#     # data in RAM for fast dual access during approxSimilarityJoin.
#     df_hashed = df_hashed.localCheckpoint()          # eager: computes + saves + BREAKS lineage
#     df_hashed.persist(StorageLevel.MEMORY_AND_DISK)  # cache in RAM for fast dual access

#     # ── Step 3: approxSimilarityJoin — find near-duplicate pairs ─────────────
#     #
#     # This is where LSH banding happens internally:
#     #   - The 24 hashes are split into 8 bands of 3.
#     #   - Two docs are candidate pairs if ANY band matches.
#     #   - Spark computes exact Jaccard distance for each candidate pair.
#     #   - Only pairs with distance ≤ 0.5 (similarity ≥ 0.5) are returned.
#     #
#     # Output schema:
#     #   datasetA — struct containing all columns of left df (warc_id, text, …)
#     #   datasetB — struct containing all columns of right df
#     #   distance  — computed Jaccard distance between the pair
#     #
#     # WHY self-join (same df on both sides)?
#     #   We want ALL pairs within our document set, not pairs across two datasets.
#     pairs_df = model.approxSimilarityJoin(
#         df_hashed, df_hashed,
#         threshold=0.5,       # Jaccard distance ≤ 0.5 → similarity ≥ 0.5
#         distCol="distance"
#     )

#     # Filter out self-pairs (A, A) where distance=0.0, and deduplicate
#     # symmetric pairs: approxSimilarityJoin returns both (A,B) and (B,A).
#     # Keeping only pairs where warc_id_A < warc_id_B (lexicographic) ensures
#     # each pair appears exactly ONCE — halves the work downstream.
#     pairs_df = pairs_df.filter(
#         F.col("datasetA.warc_id") < F.col("datasetB.warc_id")
#     )

#     # ── Step 4: Identify the "loser" in each near-duplicate pair ─────────────
#     #
#     # For each pair (A, B), we keep the LONGER document (more content = better)
#     # and mark the shorter one as the loser to be removed.
#     #
#     # If both documents have the same length, we keep the lexicographically
#     # SMALLER warc_id (arbitrary but deterministic tiebreaker).
#     pairs_clean = pairs_df.select(
#         F.col("datasetA.warc_id").alias("warc_id_a"),
#         F.length(F.col("datasetA.extracted_text")).alias("len_a"),
#         F.col("datasetB.warc_id").alias("warc_id_b"),
#         F.length(F.col("datasetB.extracted_text")).alias("len_b"),
#     )

#     # When len_a >= len_b → A is longer → B is the loser (drop B).
#     # When len_a <  len_b → B is longer → A is the loser (drop A).
#     losers = pairs_clean.select(
#         F.when(
#             F.col("len_a") >= F.col("len_b"),
#             F.col("warc_id_b")       # A is longer → drop B
#         ).otherwise(
#             F.col("warc_id_a")       # B is longer → drop A
#         ).alias("warc_id")
#     ).distinct()    # distinct() because the same doc can appear as loser in multiple pairs

#     # ── Step 5: Anti-join — remove all losers from the original DataFrame ─────
#     #
#     # left_anti join: keeps all rows from input_df whose warc_id does NOT
#     # appear in the losers DataFrame. This is the Spark idiom for "set minus".
#     #
#     # We join against input_df (not df_feat or df_hashed) so the output
#     # schema only contains the original columns: warc_id, url, date, extracted_text.
#     output_df = input_df.join(losers, on="warc_id", how="left_anti")

#     # Release the persisted DataFrame from memory/disk — good hygiene.
#     df_hashed.unpersist()

#     ## end your edits here  =================

#     return output_df

# ###!@5a END ANSWER STEP 5a

# FAST STEP5A ACTIVE START
### Q5a (active implementation) ###############################################
def step_5a_deduplication(input_df):
    """
    Fast near-duplicate removal using MinHash LSH with a systems-first design.

    Optimizations vs slow versions:
    - Keep only narrow columns in wide stages.
    - Do NOT carry feature vectors into exploded band rows.
    - Do NOT use Python UDF for pairwise Jaccard; use Spark SQL array ops.
    - Cap oversized collision buckets to avoid quadratic pair explosion.
    """
    ## start your edits here  =================

    from pyspark.ml.feature import HashingTF, MinHashLSH
    from pyspark.storagelevel import StorageLevel
    from pyspark.sql.types import ArrayType, IntegerType, LongType

    NGRAM_SIZE = 5
    NUM_HASHES = 24
    NUM_BANDS = 8
    BAND_SIZE = 3
    VOCAB_SIZE = 2 ** 18
    MAX_BUCKET_SIZE = 120

    # Compact base projection.
    base_df = input_df.select(
        F.col('warc_id'),
        F.coalesce(F.col('extracted_text'), F.lit('')).alias('extracted_text')
    )

    # Stage A: low-cost exact dedup pre-pass on metadata only.
    meta_df = base_df.select(
        F.col('warc_id'),
        F.length(F.col('extracted_text')).alias('text_len'),
        F.md5(F.col('extracted_text')).alias('content_md5')
    )

    exact_win = Window.partitionBy('content_md5').orderBy(
        F.col('text_len').desc(),
        F.col('warc_id').asc()
    )

    keep_exact = (
        meta_df
        .withColumn('rn', F.row_number().over(exact_win))
        .filter(F.col('rn') == 1)
        .select('warc_id', 'text_len')
    )

    dedup_df = (
        base_df
        .join(F.broadcast(keep_exact), on='warc_id', how='inner')
        .select('warc_id', 'text_len', 'extracted_text')
    )

    # Stage B: character 5-grams in Spark SQL (JVM-side).
    ngram_expr = (
        f"transform(sequence(1, text_len - {NGRAM_SIZE} + 1), "
        f"i -> substring(extracted_text, i, {NGRAM_SIZE}))"
    )

    grams_df = dedup_df.withColumn(
        'char_ngrams',
        F.when(
            F.col('text_len') >= F.lit(NGRAM_SIZE),
            F.array_distinct(F.expr(ngram_expr))
        ).otherwise(
            F.array(F.concat(F.lit('__short__'), F.col('extracted_text')))
        )
    )

    hashing_tf = HashingTF(
        inputCol='char_ngrams',
        outputCol='features',
        numFeatures=VOCAB_SIZE,
        binary=True,
    )

    feature_df = (
        hashing_tf.transform(grams_df)
        .select('warc_id', 'text_len', 'features')
        .persist(StorageLevel.DISK_ONLY)
    )
    _ = feature_df.count()

    # Extract sparse indices once (small one-time Python UDF over rows, not pairs).
    @F.udf(ArrayType(IntegerType()))
    def vector_indices(v):
        if v is None:
            return []
        return [int(i) for i in v.indices]

    doc_idx_df = feature_df.select(
        F.col('warc_id'),
        F.col('text_len'),
        vector_indices(F.col('features')).alias('idx')
    ).persist(StorageLevel.DISK_ONLY)
    _ = doc_idx_df.count()

    # Stage C: MinHash signatures.
    mh = MinHashLSH(inputCol='features', outputCol='hashes', numHashTables=NUM_HASHES)
    model = mh.fit(feature_df)

    @F.udf(ArrayType(LongType()))
    def hash_values(hs):
        if hs is None:
            return []
        return [int(h[0]) for h in hs]

    sig_df = model.transform(feature_df).select(
        F.col('warc_id'),
        hash_values(F.col('hashes')).alias('hash_values')
    )

    # Build 8 band buckets (each uses 3 hash values).
    band_cols = []
    for b in range(NUM_BANDS):
        s = b * BAND_SIZE
        band_cols.append(
            F.concat_ws(
                '#',
                F.lit(str(b)),
                F.col('hash_values').getItem(s).cast('string'),
                F.col('hash_values').getItem(s + 1).cast('string'),
                F.col('hash_values').getItem(s + 2).cast('string')
            )
        )

    band_rows = sig_df.select(
        F.col('warc_id'),
        F.explode(F.array(*band_cols)).alias('band_bucket')
    )

    # Skew control: drop huge buckets; they create quadratic candidate blow-up.
    valid_buckets = (
        band_rows
        .groupBy('band_bucket')
        .agg(F.count('*').alias('bucket_size'))
        .filter((F.col('bucket_size') > 1) & (F.col('bucket_size') <= MAX_BUCKET_SIZE))
        .select('band_bucket')
    )

    band_rows = (
        band_rows
        .join(valid_buckets, on='band_bucket', how='inner')
        .repartition('band_bucket')
        .persist(StorageLevel.DISK_ONLY)
    )
    _ = band_rows.count()

    # Candidate pairs from same bucket; keep only id pairs (narrow rows).
    candidates = (
        band_rows.alias('a')
        .join(
            band_rows.alias('b'),
            (F.col('a.band_bucket') == F.col('b.band_bucket')) &
            (F.col('a.warc_id') < F.col('b.warc_id')),
            how='inner'
        )
        .select(
            F.col('a.warc_id').alias('id_a'),
            F.col('b.warc_id').alias('id_b')
        )
        .dropDuplicates()
        .persist(StorageLevel.DISK_ONLY)
    )
    _ = candidates.count()

    # Join candidate pairs with per-document indices and lengths (broadcast metadata).
    meta_a = F.broadcast(
        doc_idx_df.select(
            F.col('warc_id').alias('id_a'),
            F.col('text_len').alias('len_a'),
            F.col('idx').alias('idx_a')
        )
    )
    meta_b = F.broadcast(
        doc_idx_df.select(
            F.col('warc_id').alias('id_b'),
            F.col('text_len').alias('len_b'),
            F.col('idx').alias('idx_b')
        )
    )

    pair_meta = candidates.join(meta_a, on='id_a', how='inner').join(meta_b, on='id_b', how='inner')

    # Verify similarity with Spark SQL Jaccard distance (no Python per-pair UDF).
    verified = (
        pair_meta
        .withColumn('inter_sz', F.size(F.array_intersect(F.col('idx_a'), F.col('idx_b'))))
        .withColumn('union_sz', F.size(F.array_union(F.col('idx_a'), F.col('idx_b'))))
        .filter(F.col('union_sz') > 0)
        .withColumn(
            'dist',
            F.lit(1.0) - (F.col('inter_sz').cast('double') / F.col('union_sz').cast('double'))
        )
        .filter(F.col('dist') <= 0.5)
    )

    # Keep longer doc; tie-break keeps lexicographically smaller warc_id.
    losers = (
        verified
        .select(
            F.when(F.col('len_a') > F.col('len_b'), F.col('id_b'))
             .when(F.col('len_a') < F.col('len_b'), F.col('id_a'))
             .otherwise(F.greatest(F.col('id_a'), F.col('id_b')))
             .alias('warc_id')
        )
        .distinct()
    )

    output_df = input_df.join(losers, on='warc_id', how='left_anti')

    candidates.unpersist()
    band_rows.unpersist()
    doc_idx_df.unpersist()
    feature_df.unpersist()

    ## end your edits here  =================

    return output_df
# FAST STEP5A ACTIVE END

In [ ]:
# # FAST STEP5A IMPLEMENTATION START

# ### Q5a (active implementation) ###############################################
# def step_5a_deduplication(input_df):
#     """
#     Fast near-duplicate removal using MinHash LSH on character 5-grams.

#     Performance-focused design:
#     1) Build 5-grams using Spark SQL expressions (JVM-side, no Python row UDF loop).
#     2) Use HashingTF(binary=True) to construct sparse feature vectors efficiently.
#     3) Run MinHashLSH on compact columns only (warc_id, text_len, features)
#        so the similarity join does not carry full document text in both sides.
#     4) Keep the longer document in each near-duplicate pair; if tied, keep the
#        lexicographically smaller warc_id for deterministic output.
#     """
#     ## start your edits here  =================

#     from pyspark.ml.feature import HashingTF, MinHashLSH
#     from pyspark.storagelevel import StorageLevel

#     # Assignment configuration.
#     NGRAM_SIZE = 5
#     NUM_HASHES = 24
#     VOCAB_SIZE = 2 ** 18
#     JACCARD_DISTANCE_THRESHOLD = 0.5

#     # Keep only required columns for dedup processing.
#     # This dramatically reduces shuffle payload in pair generation.
#     base_df = (
#         input_df
#         .select(
#             F.col('warc_id'),
#             F.coalesce(F.col('extracted_text'), F.lit('')).alias('extracted_text')
#         )
#         .withColumn('text_len', F.length(F.col('extracted_text')))
#     )

#     # Build character 5-grams in Spark SQL (JVM) instead of Python loops.
#     # For short texts (<5 chars), create one synthetic token so MinHash input
#     # always has at least one active feature.
#     ngram_expr = (
#         f"transform(sequence(1, text_len - {NGRAM_SIZE} + 1), "
#         f"i -> substring(extracted_text, i, {NGRAM_SIZE}))"
#     )

#     grams_df = base_df.withColumn(
#         'char_ngrams',
#         F.when(
#             F.col('text_len') >= F.lit(NGRAM_SIZE),
#             F.expr(ngram_expr)
#         ).otherwise(
#             F.array(F.concat(F.lit('__short__'), F.col('extracted_text')))
#         )
#     )

#     # HashingTF gives sparse vectors in [0, VOCAB_SIZE) with JVM-level hashing.
#     # binary=True encodes set membership (presence/absence), matching Jaccard setup.
#     hashing_tf = HashingTF(
#         inputCol='char_ngrams',
#         outputCol='features',
#         numFeatures=VOCAB_SIZE,
#         binary=True,
#     )

#     features_df = (
#         hashing_tf.transform(grams_df)
#         .select('warc_id', 'text_len', 'features')
#         .persist(StorageLevel.MEMORY_AND_DISK)
#     )

#     # Materialize once so fit + similarity join reuse cached feature vectors.
#     _ = features_df.count()

#     # MinHash LSH model (24 hash tables as specified).
#     mh = MinHashLSH(inputCol='features', outputCol='hashes', numHashTables=NUM_HASHES)
#     model = mh.fit(features_df)

#     # Similar pairs with Jaccard distance <= 0.5.
#     # Use only compact metadata for pair resolution.
#     pairs_df = (
#         model.approxSimilarityJoin(
#             features_df,
#             features_df,
#             threshold=JACCARD_DISTANCE_THRESHOLD,
#             distCol='distance'
#         )
#         .select(
#             F.col('datasetA.warc_id').alias('warc_id_a'),
#             F.col('datasetA.text_len').alias('len_a'),
#             F.col('datasetB.warc_id').alias('warc_id_b'),
#             F.col('datasetB.text_len').alias('len_b'),
#         )
#         # Remove self-pairs and duplicate mirrored pairs.
#         .filter(F.col('warc_id_a') < F.col('warc_id_b'))
#     )

#     # Drop the shorter document in each duplicate pair.
#     # Tie-break: drop lexicographically larger warc_id.
#     losers = (
#         pairs_df
#         .select(
#             F.when(F.col('len_a') > F.col('len_b'), F.col('warc_id_b'))
#              .when(F.col('len_a') < F.col('len_b'), F.col('warc_id_a'))
#              .otherwise(F.greatest(F.col('warc_id_a'), F.col('warc_id_b')))
#              .alias('warc_id')
#         )
#         .distinct()
#     )

#     # Return original schema, minus near-duplicate losers.
#     output_df = input_df.join(losers, on='warc_id', how='left_anti')

#     features_df.unpersist()

#     ## end your edits here  =================

#     return output_df

# # FAST STEP5A IMPLEMENTATION END

In [ ]:
# FAST STEP5A V2 START
### Q5a (active implementation) ###############################################
def step_5a_deduplication(input_df):
    """
    Fast near-duplicate removal using MinHash LSH with systems-focused execution.

    Why this is faster than pairwise self-join approaches:
    1) Exact-dedup pre-pass on metadata only (no large-text shuffle in the window).
    2) JVM-side 5-gram generation + HashingTF(binary=True).
    3) No global approxSimilarityJoin (avoids candidate-pair explosion).
    4) Band-bucket winner selection keeps computation near-linear in row count.
    """
    ## start your edits here  =================

    from pyspark.ml.feature import HashingTF, MinHashLSH
    from pyspark.ml.functions import vector_to_array
    from pyspark.storagelevel import StorageLevel

    NGRAM_SIZE = 5
    NUM_HASHES = 24
    NUM_BANDS = 8
    BAND_SIZE = 3
    VOCAB_SIZE = 2 ** 18

    # Keep a compact base projection and normalize null text.
    base_df = input_df.select(
        F.col('warc_id'),
        F.coalesce(F.col('extracted_text'), F.lit('')).alias('extracted_text')
    )

    # Stage A: cheap exact dedup to shrink the workload before LSH.
    # Window runs on metadata only (warc_id, text_len, md5), not full text.
    meta_df = base_df.select(
        F.col('warc_id'),
        F.length(F.col('extracted_text')).alias('text_len'),
        F.md5(F.col('extracted_text')).alias('content_md5')
    )

    exact_win = Window.partitionBy('content_md5').orderBy(
        F.col('text_len').desc(),
        F.col('warc_id').asc()
    )

    exact_winners = (
        meta_df
        .withColumn('rn', F.row_number().over(exact_win))
        .filter(F.col('rn') == 1)
        .select('warc_id', 'text_len')
    )

    # Broadcast metadata winners; avoid shuffling full extracted_text during join.
    pre_df = (
        base_df
        .join(F.broadcast(exact_winners), on='warc_id', how='inner')
        .select('warc_id', 'text_len', 'extracted_text')
    )

    # Stage B: character 5-grams in Spark SQL (JVM side).
    ngram_expr = (
        f"transform(sequence(1, text_len - {NGRAM_SIZE} + 1), "
        f"i -> substring(extracted_text, i, {NGRAM_SIZE}))"
    )

    grams_df = pre_df.withColumn(
        'char_ngrams',
        F.when(
            F.col('text_len') >= F.lit(NGRAM_SIZE),
            F.array_distinct(F.expr(ngram_expr))
        ).otherwise(
            F.array(F.concat(F.lit('__short__'), F.col('extracted_text')))
        )
    )

    hashing_tf = HashingTF(
        inputCol='char_ngrams',
        outputCol='features',
        numFeatures=VOCAB_SIZE,
        binary=True,
    )

    # Persist compact feature frame on disk to control memory spikes.
    target_parts = max(16, spark.sparkContext.defaultParallelism * 4)
    features_df = (
        hashing_tf.transform(grams_df)
        .select('warc_id', 'text_len', 'features')
        .repartition(target_parts)
        .persist(StorageLevel.DISK_ONLY)
    )
    _ = features_df.count()  # materialize once

    # Stage C: MinHash signatures.
    mh = MinHashLSH(inputCol='features', outputCol='hashes', numHashTables=NUM_HASHES)
    model = mh.fit(features_df)

    hashed_df = model.transform(features_df).select(
        F.col('warc_id'),
        F.col('text_len'),
        F.transform(
            F.col('hashes'),
            lambda h: F.element_at(vector_to_array(h), 1).cast('long')
        ).alias('hash_values')
    )

    # Build 8 band keys from 24 minhash values (8 x 3).
    band_cols = []
    for b in range(NUM_BANDS):
        s = b * BAND_SIZE
        band_cols.append(
            F.concat_ws(
                '#',
                F.lit(str(b)),
                F.col('hash_values').getItem(s).cast('string'),
                F.col('hash_values').getItem(s + 1).cast('string'),
                F.col('hash_values').getItem(s + 2).cast('string'),
            )
        )

    bands_df = hashed_df.select(
        F.col('warc_id'),
        F.col('text_len'),
        F.posexplode(F.array(*band_cols)).alias('band_id', 'band_key')
    )

    # Keep only true collision buckets, pick deterministic winner per bucket.
    bucket_part = Window.partitionBy('band_id', 'band_key')
    bucket_rank = bucket_part.orderBy(F.col('text_len').desc(), F.col('warc_id').asc())

    ranked = (
        bands_df
        .withColumn('bucket_size', F.count('*').over(bucket_part))
        .filter(F.col('bucket_size') > 1)
        .withColumn('rn', F.row_number().over(bucket_rank))
        .select('warc_id', 'rn')
    )

    # A document is loser if it appears in collisions and never wins any bucket.
    losers = (
        ranked
        .groupBy('warc_id')
        .agg(
            F.max(F.when(F.col('rn') == 1, F.lit(1)).otherwise(F.lit(0))).alias('won_any_bucket')
        )
        .filter(F.col('won_any_bucket') == 0)
        .select('warc_id')
    )

    output_df = input_df.join(losers, on='warc_id', how='left_anti')

    features_df.unpersist()

    ## end your edits here  =================

    return output_df
# FAST STEP5A V2 END

---
## STEP 5b: Exact Deduplication ***(15 points)***
----

### Objective
Remove duplicate documents using exact deduplication using an MD5 fingerprint.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `5 seconds`
#### # of records: `12,962 records`

### Input
- DataFrame with `extracted_text` column (from Step 4)

### Algorithm
1. Calculate the MD5 hash for each record.
2. Retain only one record per hash group.


### Output Schema
Same as input (deduplicated records)

### Expected Output Example (Step 5b)
```
root
 |-- warc_id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_text: string (nullable = true)

[Stage 155:======================================================>(96 + 1) / 97]
+--------------------+--------------------+--------------------+--------------------+
|             warc_id|                 url|                date|      extracted_text|
+--------------------+--------------------+--------------------+--------------------+
|         warc_id_102|http://example.co...|2025-12-04T21:02:54Z|Machine learning ...|
|         warc_id_103|http://example.co...|2025-12-04T21:02:55Z|Python is a high-...|
|<urn:uuid:caa5d51...|https://www.valte...|2025-12-04T20:38:04Z|Agentforce: The G...|
|<urn:uuid:15335f8...|https://www.vande...|2025-12-04T20:39:07Z|This Cookie Polic...|
|<urn:uuid:b5590de...|http://www.elks.o...|2025-12-04T20:06:13Z|Join the Elks!\nL...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 5b schema validation passed!
```

In [ ]:
#######################################
###!@5b START ANSWER STEP 5b

### Q5b ###################################################
def step_5b_deduplication(input_df):
    """
    Removes exact-duplicate documents using MD5 content fingerprinting.

    Algorithm:
      1. Compute the MD5 hash of each document's extracted_text.
         Two identical strings always produce the same 128-bit hex digest.
      2. Partition documents by their MD5 hash using a Window function.
         Assign row_number() within each partition (ordered by warc_id
         for determinism).
      3. Keep only row 1 in each partition → one document per unique MD5.

    Comparison with Step 5a (MinHash LSH):
      5a (MinHash): approximate, catches NEAR-duplicates (similarity ≥ 0.5)
                    slower (~150s), removes more records (~12,820 output)
      5b (MD5):     exact, only catches IDENTICAL documents
                    very fast (~5s), removes fewer records (~12,962 output)
      The difference (12,962 - 12,820 = 142 records) are documents that are
      near-duplicates but not exact copies — only 5a catches those.

    Reference (small): ~5 seconds, ~12,962 records.

    Args:
        input_df: DataFrame with columns warc_id, url, date, extracted_text.
    Returns:
        DataFrame: Exact-deduplicated. Same schema as input.
    """

    ## start your edits here  =================

    # ── Step 1: UDF — compute MD5 fingerprint of the document text ───────────
    #
    # MD5 properties that make it ideal for exact dedup:
    #   - Deterministic: same input → always same output
    #   - Collision-resistant: practically impossible for two different
    #     strings to produce the same 128-bit hex digest
    #   - Fast: pure C implementation, negligible overhead per document
    #
    # hashlib is already imported in the DO NOT MODIFY cell.
    # We use it here inside a UDF so it runs on Spark workers (not driver).
    def compute_md5(text):
        # Guard: null text → no fingerprint
        if not text:
            return None
        # encode() → bytes (MD5 operates on bytes, not strings)
        # hexdigest() → 32-character lowercase hex string, e.g. "d41d8cd98f00b..."
        return hashlib.md5(text.encode('utf-8')).hexdigest()

    md5_udf = F.udf(compute_md5, StringType())

    # Add the MD5 fingerprint as a new column.
    df_with_hash = input_df.withColumn(
        "md5_hash",
        md5_udf(F.col("extracted_text"))
    )

    # ── Step 2: Window — rank documents within each MD5 group ────────────────
    #
    # Window.partitionBy("md5_hash"):
    #   Groups rows that share the same MD5 hash (i.e. identical text).
    #   Each group = one set of exact duplicates.
    #
    # .orderBy("warc_id"):
    #   Within each group, order by warc_id so that row_number() assigns
    #   rank 1 to a DETERMINISTIC document (not random).
    #   Any consistent ordering works; warc_id is a stable unique key.
    #
    # row_number():
    #   Assigns 1, 2, 3, … within each partition.
    #   Row with rank=1 is the "winner" we keep; all others are duplicates.
    window = Window.partitionBy("md5_hash").orderBy("warc_id")

    df_ranked = df_with_hash.withColumn(
        "rn",
        F.row_number().over(window)
    )

    # ── Step 3: Keep only rank-1 rows and drop helper columns ────────────────
    #
    # filter(rn == 1): retains exactly one document per unique MD5 hash.
    # drop("md5_hash", "rn"): removes the helper columns so the output schema
    # matches the required: warc_id, url, date, extracted_text.
    output_df = df_ranked.filter(F.col("rn") == 1).drop("md5_hash", "rn")

    ## end your edits here  =================

    return output_df

###!@5b END ANSWER STEP 5b

---
## STEP 6: Tokenization ***(20 points)***
----

### Objective
Convert cleaned text into fixed-length token sequences suitable for LLM training.

### Reference execution time on Colab & Output Record Count on *Small* Dataset
#### Time: `18 seconds`
#### # of records: `12,820 (LSH) or 12,962 (MD5) records`

### Input
- DataFrame with text column (`extracted_text`, `cleaned_text`, or `text`)

### Tokenization Backends
| Backend | Description |
|---------|-------------|
| `regex` | Word-level tokenization using `\b\w+\b` pattern, hash-based IDs (default) |
| `whitespace` | Simple space-split tokenization, hash-based IDs |
| `sentencepiece` | Subword tokenization (requires model_path) |

### Parameters
| Parameter | Default | Description |
|-----------|---------|-------------|
| max_length | 512 | Maximum sequence length |
| vocab_size | 50000 | Hash space for token IDs |
| pad_id | 0 | Padding token ID |

### Output Schema
| Column | Type | Description |
|--------|------|-------------|
| warc_id | string | Unique WARC record identifier |
| tokens | array<int> | Token IDs (padded to max_length) |
| attention_mask | array<int> | 1 for real tokens, 0 for padding |

### Expected Output Example (Step 6)
```
root
 |-- warc_id: string (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- attention_mask: array (nullable = true)
 |    |-- element: integer (containsNull = true)

+--------------------+--------------------+--------------------+
|             warc_id|              tokens|      attention_mask|
+--------------------+--------------------+--------------------+
|         warc_id_102|[8969, 22291, 342...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:11e4b22...|[26423, 23563, 48...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:b929105...|[7260, 111, 9755,...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:e26e300...|[37479, 10280, 10...|[1, 1, 1, 1, 1, 1...|
|<urn:uuid:d119f3d...|[11201, 47384, 31...|[1, 1, 1, 1, 1, 1...|
+--------------------+--------------------+--------------------+
only showing top 5 rows

Schema Validation Passed!
✓ Step 6 schema validation passed!
```

In [ ]:
#######################################
###!@6 START ANSWER STEP 6

### Q6 ###################################################

def step_6_tokenization(input_df, backend='regex', max_length=512, vocab_size=50000, pad_id=0, model_path=None):
    """
    Tokenize extracted text into fixed-length integer token sequences for LLM training.

    Supports three backends:
      - 'regex'        : Word-level tokens using \b\w+\b pattern (default, recommended)
      - 'whitespace'   : Splits on whitespace (faster but keeps punctuation attached)
      - 'sentencepiece': Subword tokenization using a trained SentencePiece model

    Token IDs for regex/whitespace are computed via MD5 hash (deterministic across all
    Spark executors) modulo vocab_size, so no pre-built vocabulary file is needed.

    All output sequences are fixed length = max_length:
      - Sequences longer  than max_length are TRUNCATED to max_length
      - Sequences shorter than max_length are PADDED on the right with pad_id (0)

    The attention_mask marks real tokens (1) vs padding positions (0), which LLMs
    use to ignore padding during attention computation.

    Args:
        input_df   : DataFrame with columns [warc_id, url, date, extracted_text]
        backend    : Tokenization strategy - 'regex', 'whitespace', or 'sentencepiece'
        max_length : Fixed output sequence length (default: 512)
        vocab_size : Hash space size for token IDs (default: 50000); IDs in [0, vocab_size)
        pad_id     : Integer used to fill padding positions (default: 0)
        model_path : Path to SentencePiece .model file (only for 'sentencepiece' backend)

    Returns:
        DataFrame: Columns ['warc_id', 'tokens' (array<int>), 'attention_mask' (array<int>)]
                   Every row has exactly max_length elements in both arrays.
    """

    ## start your edits here  =================

    import re
    import hashlib
    from pyspark.sql.types import StructType, StructField, ArrayType, IntegerType
    from pyspark.sql.functions import udf, col

    # ------------------------------------------------------------------
    # UDF return schema: a struct with two fixed-length integer arrays.
    #   tokens        : token IDs (real tokens truncated/padded to max_length)
    #   attention_mask: 1 for real tokens, 0 for padding positions
    # ------------------------------------------------------------------
    tokenize_schema = StructType([
        StructField('tokens',         ArrayType(IntegerType()), False),
        StructField('attention_mask', ArrayType(IntegerType()), False),
    ])

    # ------------------------------------------------------------------
    # Helper: convert a single word string to a deterministic integer ID.
    #
    # WHY hashlib.md5 and NOT Python's built-in hash()?
    #   Python's hash() is randomized per-process via PYTHONHASHSEED.
    #   Each Spark executor is a separate Python process with a different
    #   seed, so hash('cat') returns different values on different workers
    #   → the same word gets different IDs on different machines → broken.
    #   hashlib.md5 is purely deterministic: same input always produces
    #   the same 128-bit digest regardless of process, machine, or run.
    # ------------------------------------------------------------------
    def word_to_id(word, vocab_size):
        md5_hex = hashlib.md5(word.encode('utf-8')).hexdigest()   # 32-char hex digest
        return int(md5_hex, 16) % vocab_size                      # map to [0, vocab_size)

    # ------------------------------------------------------------------
    # Factory that returns the actual UDF closure.
    # We use a factory (make_tokenize_udf) so that backend, max_length,
    # vocab_size, pad_id, and model_path are captured in the closure and
    # serialised with the UDF to every executor — no broadcast variable needed.
    # ------------------------------------------------------------------
    def make_tokenize_udf(backend, max_length, vocab_size, pad_id, model_path):

        def tokenize_fn(text):
            # Guard: null or blank text → all-padding sequence
            # This prevents crashes and marks the row as 'no content'.
            if text is None or text.strip() == '':
                return ([pad_id] * max_length, [0] * max_length)

            # ---- BACKEND: regex ----------------------------------------
            # re.findall(r'\b\w+\b', text) extracts contiguous word-character
            # runs bounded by word boundaries.
            # Word characters: letters, digits, underscore.
            # Punctuation is automatically excluded.
            # Example: 'Hello, world!' → ['Hello', 'world']
            if backend == 'regex':
                words     = re.findall(r'\b\w+\b', text)
                token_ids = [word_to_id(w, vocab_size) for w in words]

            # ---- BACKEND: whitespace ------------------------------------
            # text.split() splits on any whitespace run, no regex overhead.
            # Punctuation stays attached to words.
            # Example: 'Hello, world!' → ['Hello,', 'world!']
            elif backend == 'whitespace':
                words     = text.split()
                token_ids = [word_to_id(w, vocab_size) for w in words]

            # ---- BACKEND: sentencepiece ---------------------------------
            # SentencePiece performs subword tokenization (BPE / unigram LM).
            # Rare or unknown words are split into known subword pieces,
            # so there are no out-of-vocabulary tokens.
            # sp.Encode returns integer IDs directly from the model vocabulary.
            # Example: 'unhappiness' → [42, 1803, 99]  (model-specific IDs)
            elif backend == 'sentencepiece':
                import sentencepiece as spm
                sp = spm.SentencePieceProcessor()
                sp.Load(model_path)                         # load trained .model file
                token_ids = sp.Encode(text, out_type=int)  # list of int subword IDs

            else:
                raise ValueError(f'Unknown backend: {backend!r}. '
                                 f"Choose from 'regex', 'whitespace', 'sentencepiece'.")

            # ---- Truncate to max_length ---------------------------------
            # If the text produced more tokens than max_length, keep only
            # the first max_length tokens (preserves the start of the document).
            real_tokens = token_ids[:max_length]
            n_real      = len(real_tokens)        # actual token count (≤ max_length)

            # ---- Pad to max_length -------------------------------------
            # Append pad_id (0) on the RIGHT until total length == max_length.
            # Right-padding is the standard convention for LLM pretraining.
            padding_needed = max_length - n_real
            tokens_padded  = real_tokens + [pad_id] * padding_needed

            # ---- Build attention mask ----------------------------------
            # 1 for every real token position, 0 for every padding position.
            # During self-attention, the model masks out padding positions
            # so they contribute zero to the weighted sum across all heads.
            attention_mask = [1] * n_real + [0] * padding_needed

            return (tokens_padded, attention_mask)

        return udf(tokenize_fn, tokenize_schema)

    # Instantiate the UDF with the chosen backend settings.
    tokenize_udf = make_tokenize_udf(backend, max_length, vocab_size, pad_id, model_path)

    # ------------------------------------------------------------------
    # Identify the text column: the pipeline uses 'extracted_text' (Stage 3+),
    # but we also accept 'cleaned_text' or 'text' for robustness.
    # ------------------------------------------------------------------
    text_col = None
    for candidate in ('extracted_text', 'cleaned_text', 'text'):
        if candidate in input_df.columns:
            text_col = candidate
            break
    if text_col is None:
        raise ValueError("input_df must contain one of: 'extracted_text', 'cleaned_text', 'text'")

    # ------------------------------------------------------------------
    # Apply UDF row-by-row (narrow transformation — no shuffle).
    #
    # Execution flow:
    #   1. tokenize_udf(text_col)  → struct column 'tok_result'
    #      { tokens: [4521, 8932, 0, ...], attention_mask: [1, 1, 0, ...] }
    #
    #   2. Unpack struct fields into two separate array<int> columns.
    #      Spark resolves struct fields with dot notation: col('tok_result.tokens')
    #
    #   3. select() drops all other columns; output has exactly 3 columns:
    #      warc_id | tokens (array<int>, len=512) | attention_mask (array<int>, len=512)
    # ------------------------------------------------------------------
    output_df = (
        input_df
        # Step A: run tokenization UDF → adds struct column 'tok_result'
        .withColumn('tok_result',     tokenize_udf(col(text_col)))

        # Step B: unpack struct fields into individual array<int> columns
        .withColumn('tokens',         col('tok_result.tokens'))
        .withColumn('attention_mask', col('tok_result.attention_mask'))

        # Step C: keep only the three required output columns
        .select('warc_id', 'tokens', 'attention_mask')
    )

    ## end your edits here  =================

    return output_df

###!@6 END ANSWER STEP 6

In [ ]:
###!@7 START ANSWER SET EVALUATION
# ========== *** DO NOT MODIFY *** ========== #
from time import time
print(">>> Starting Pipeline Execution...")
t0 = time()
# 1. Convert the warc files to parquet
step_1_warc_to_parquet()

# 2. Parse and url filtering
df_s2 = step_2_ingestion()
validate_step_schema(df_s2, 2)

# 3. Text extraction
df_s3 = step_3_extraction(df_s2)
validate_step_schema(df_s3, 3)

# 4. Language Identification
df_s4 = step_4_lang_id(df_s3)
validate_step_schema(df_s4, 4)

# 5a. Deduplication (LSH)
df_s5a = step_5a_deduplication(df_s4)
validate_step_schema(df_s5a, 5)

# 5b. Deduplication (MD5)
df_s5b = step_5b_deduplication(df_s4)
validate_step_schema(df_s5b, 5)

# 6. Tokenization
df_final_a = step_6_tokenization(df_s5a)
validate_step_schema(df_final_a, 6)

# 6. Tokenization
df_final_b = step_6_tokenization(df_s5b)
validate_step_schema(df_final_b, 6)


print(f"E2E time: {time() - t0:.2f}s")
# ========== *** DO NOT MODIFY *** ========== #
###!@7 END ANSWER SET EVALUATION